# Lesson 2.2 — Geometric Jacobian by Column Construction (Revolute)
**Module 6 · Unit 2 · Lesson 6**

Each revolute column is $[\,\mathbf z\times(\mathbf o_n-\mathbf o);\,\mathbf z\,]$ — read straight off the arm's geometry. We build it from the forward-kinematics frames, match the planar 2R symbolic $J_v$, and validate a spatial 3R arm against finite differences.

In [1]:
import numpy as np

def skew(v):
    v=np.asarray(v,float).ravel()
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])

def dh(theta,d,a,alpha):
    # one Denavit-Hartenberg link transform
    ct,st,ca,sa=np.cos(theta),np.sin(theta),np.cos(alpha),np.sin(alpha)
    return np.array([[ct,-st*ca, st*sa, a*ct],
                     [st, ct*ca,-ct*sa, a*st],
                     [0,  sa,    ca,    d   ],
                     [0,  0,     0,     1   ]])

def forward_chain(params,jtypes,q):
    # returns T_0^0..T_0^n ; revolute adds q to theta, prismatic adds q to d
    T=np.eye(4); Ts=[T.copy()]
    for i,(th0,d0,a,al) in enumerate(params):
        th,d=(th0+q[i],d0) if jtypes[i]=="R" else (th0,d0+q[i])
        T=T@dh(th,d,a,al); Ts.append(T.copy())
    return Ts

def geometric_jacobian(params,jtypes,q):
    # base-frame 6xn geometric Jacobian (D-057: [v; omega], linear on top)
    Ts=forward_chain(params,jtypes,q); o_n=Ts[-1][:3,3]; n=len(q); J=np.zeros((6,n))
    for i in range(n):
        z=Ts[i][:3,2]; o=Ts[i][:3,3]   # axis & origin of frame i (= z_{i-1}, o_{i-1})
        if jtypes[i]=="R": J[:3,i]=np.cross(z,o_n-o); J[3:,i]=z   # revolute: [z x (o_n-o); z]
        else:             J[:3,i]=z;                   J[3:,i]=0   # prismatic: [z; 0]
    return J

def fk_pose(params,jtypes,q):
    T=forward_chain(params,jtypes,q)[-1]; return T[:3,3], T[:3,:3]


## Revolute columns reproduce the planar 2R worked example

In [2]:
checks=[]
L1=L2=1.0; params=[(0,0,L1,0),(0,0,L2,0)]; jtypes=["R","R"]
Ts=forward_chain(params,jtypes,np.array([0.0,np.pi/2]))
o_n=Ts[-1][:3,3]
col1=np.cross(Ts[0][:3,2], o_n-Ts[0][:3,3])   # z0 x (o_n - o0)
col2=np.cross(Ts[1][:3,2], o_n-Ts[1][:3,3])   # z1 x (o_n - o1)
print("column 1 linear:",col1,"  column 2 linear:",col2)
checks.append(np.allclose(col1,[-1,1,0]) and np.allclose(col2,[-1,0,0]))

column 1 linear: [-1.  1.  0.]   column 2 linear: [-1.  0.  0.]


## Match symbolic $J_v$ across several poses

In [3]:
def Jv_symbolic(th):
    t1,t2=th; s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12,-L2*s12],[L1*c1+L2*c12,L2*c12]])
for th in [np.array([0,np.pi/2]),np.array([0.3,0.7]),np.array([1.1,-0.4])]:
    ok=np.allclose(geometric_jacobian(params,jtypes,th)[:2,:],Jv_symbolic(th))
    checks.append(ok)
print("all poses match symbolic:",all(checks[-3:]))

all poses match symbolic: True


## Spatial 3R arm: full 6×3 geometric J vs finite differences

In [4]:
L2_,L3_=1.0,0.8
p3=[(0,0,0,np.pi/2),(0,0,L2_,0),(0,0,L3_,0)]; t3=["R","R","R"]
def fd_jac(params,jtypes,q,eps=1e-6):
    n=len(q); J=np.zeros((6,n))
    for i in range(n):
        e=np.zeros(n); e[i]=eps
        pp,Rp=fk_pose(params,jtypes,q+e); pm,Rm=fk_pose(params,jtypes,q-e)
        J[:3,i]=(pp-pm)/(2*eps); M=Rp@Rm.T; J[3:,i]=vee((M-M.T)/2)/(2*eps)
    return J
for q in [np.array([0.2,0.5,-0.3]),np.array([1.0,-0.6,0.9])]:
    ok=np.allclose(geometric_jacobian(p3,t3,q),fd_jac(p3,t3,q),atol=1e-5); checks.append(ok)
print("spatial 3R geometric == finite-difference:",all(checks[-2:]))

spatial 3R geometric == finite-difference: True


In [5]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
